<a href="https://colab.research.google.com/github/leobrewer10/MSBA-PIPELINES/blob/main/weather_golf_ETL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import logging
from sqlalchemy import create_engine

"""
Louisville Golf Tee Time Weather Advisor

Author: Leo Brewer
Data Source: Open-Meteo API

"""


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logging.info("Starting Golf Weather ETL Pipeline")


def extract_weather_data():
    """
    Extract weather forecast data from Open-Meteo API
    """

    url = (
        "https://api.open-meteo.com/v1/forecast"
        "?latitude=38.2527"
        "&longitude=-85.7585"
        "&hourly=temperature_2m,precipitation_probability,"
        "wind_speed_10m"
        "&forecast_days=3"
    )

    try:
        response = requests.get(url)
        response.raise_for_status()

        logging.info("Weather API extraction successful")

        return response.json()

    except Exception as e:
        logging.error(f"Weather API extraction failed: {e}")
        raise

def transform_weather_data(raw_data):
    """
    Transform Open-Meteo JSON into a Pandas DataFrame
    """

    hourly = raw_data["hourly"]

    df = pd.DataFrame({
        "forecast_time": hourly["time"],
        "temperature": hourly["temperature_2m"],
        "rain_probability": hourly["precipitation_probability"],
        "wind_speed": hourly["wind_speed_10m"]
    })

    logging.info("Weather data transformed into DataFrame")

    return df


def clean_weather_data(df):
    """
    Clean and standardize weather data
    """

    # Convert forecast_time to datetime
    df["forecast_time"] = pd.to_datetime(df["forecast_time"])

    # Remove duplicate records
    df = df.drop_duplicates()

    # Fill missing values
    df["rain_probability"] = df["rain_probability"].fillna(0)
    df["wind_speed"] = df["wind_speed"].fillna(0)

    logging.info("Data cleaning completed")

    return df


def validate_data(df):
    """
    Perform data quality validation checks
    """

    logging.info("Running data validation checks")

    # Null value check
    null_count = df.isnull().sum().sum()

    if null_count == 0:
        logging.info("PASS: No null values found")
    else:
        logging.warning(f"WARNING: {null_count} null values found")

    # Duplicate check
    duplicate_count = df.duplicated().sum()

    if duplicate_count == 0:
        logging.info("PASS: No duplicate records found")
    else:
        logging.warning(f"WARNING: {duplicate_count} duplicate records found")

    # Row count check
    logging.info(f"Rows available for processing: {len(df)}")

    return True

def create_golf_metrics(df):
    """
    Create golf-specific analytics metrics
    """

    # Cancellation Probability
    df["cancellation_probability"] = df["rain_probability"]

    # Playability Score
    df["playability_score"] = 100

    # Rain penalties
    df.loc[df["rain_probability"] > 20, "playability_score"] -= 15
    df.loc[df["rain_probability"] > 40, "playability_score"] -= 25
    df.loc[df["rain_probability"] > 70, "playability_score"] -= 35

    # Wind penalties
    df.loc[df["wind_speed"] > 10, "playability_score"] -= 10
    df.loc[df["wind_speed"] > 20, "playability_score"] -= 20

    # Prevent negative scores
    df["playability_score"] = df["playability_score"].clip(lower=0)

    # Recommended Round
    df["recommended_round"] = "18 Holes"

    df.loc[
        (df["rain_probability"] > 50) |
        (df["wind_speed"] > 20),
        "recommended_round"
    ] = "9 Holes"

    logging.info("Golf recommendation metrics created")

    return df

def incremental_load_check(df):
    """
    Prevent duplicate forecast records
    based on forecast_time
    """

    original_rows = len(df)

    df = df.drop_duplicates(subset=["forecast_time"])

    final_rows = len(df)

    logging.info(
        f"Incremental load check completed. "
        f"Removed {original_rows - final_rows} duplicates."
    )

    return df

# --- ETL Pipeline Execution ---
raw_data = extract_weather_data()

weather_df = transform_weather_data(raw_data)

weather_df = clean_weather_data(weather_df)

weather_df = incremental_load_check(weather_df) # Apply incremental load check after cleaning

validate_data(weather_df) # Call the validation function

weather_df = create_golf_metrics(weather_df) # Apply golf metrics

# Final data quality check
# Rain probability range check
if (weather_df["rain_probability"].between(0, 100)).all():
    logging.info("PASS: Rain probability values valid")
else:
    logging.warning("WARNING: Invalid rain probability values found")

logging.info(f"Rows extracted: {len(weather_df)}")
print(weather_df.head())
print(weather_df.shape)

analytics_df = weather_df[
    [
        "forecast_time",
        "temperature",
        "rain_probability",
        "wind_speed",
        "playability_score",
        "cancellation_probability",
        "recommended_round"
    ]
]

analytics_df.to_csv(
    "golf_weather_analytics.csv",
    index=False
)

logging.info("Analytics dataset exported to CSV")

# Temperature validation

if (
    weather_df["temperature"].between(-50, 130)
).all():
    logging.info(
        "PASS: Temperature values valid"
    )
else:
    logging.warning(
        "WARNING: Temperature values outside expected range"
    )



# functions (these lines seem to be a leftover or a mistaken re-declaration)

        forecast_time  temperature  rain_probability  wind_speed  \
0 2026-06-12 00:00:00         28.7                 1        11.0   
1 2026-06-12 01:00:00         27.9                 0        10.7   
2 2026-06-12 02:00:00         27.0                 0         6.3   
3 2026-06-12 03:00:00         26.2                 0         6.3   
4 2026-06-12 04:00:00         25.9                 1         5.8   

   cancellation_probability  playability_score recommended_round  
0                         1                 90          18 Holes  
1                         0                 90          18 Holes  
2                         0                100          18 Holes  
3                         0                100          18 Holes  
4                         1                100          18 Holes  
(72, 7)
